<center><h1>VODAPHONE AGE DATASET ANALYSIS<h1><center>

In [961]:
import pandas as pd

In [962]:
df = pd.read_csv('./vodafone_age_subset.csv', sep=",")
df.head()

,CALCULATION_METHOD_ID,calls_count_in_weekdays,calls_duration_in_weekdays,calls_count_out_weekdays,calls_duration_out_weekdays,calls_count_in_weekends,calls_duration_in_weekends,calls_count_out_weekends,calls_duration_out_weekends,DATA_VOLUME_WEEKDAYS,...,applemaps_volume,applemaps_count,msoffice365_volume,msoffice365_count,jabber_volume,jabber_count,telegram_volume,telegram_count,user_hash,target
0,2,10.87,32.025,17.74,40.819,7.00,21.463,11.13,20.427,154.837,...,0.0,0.0,0.09,9.29,0.0000,0.0,0.0,0.0,312ca09052ac6eb49fbd2a546a782df5,4
1,1,0.91,1.346,0.48,0.546,0.25,0.688,0.75,0.708,53.639,...,0.0,0.0,0.00,0.00,0.0033,0.1,0.0,0.0,327733ea2cea082b48707d2700b49327,4
2,2,0.00,0.000,0.00,0.000,0.00,0.000,0.00,0.000,0.000,...,0.0,0.0,0.00,0.00,0.0000,0.0,0.0,0.0,22ede60385359c8c24bc68449ca56763,6
3,2,7.39,30.278,11.00,44.182,5.00,20.008,8.38,32.385,0.000,...,0.0,0.0,0.00,0.00,0.0000,0.0,0.0,0.0,67682b3b1d1319a7cf90ed80eb16b899,5
4,1,0.43,0.996,0.70,1.609,2.25,3.115,0.00,0.000,0.000,...,0.0,0.0,0.00,0.00,0.0000,0.0,0.0,0.0,334e5ceddcc4f11f261322832526ae49,3


In [963]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21000 entries, 0 to 20999
Columns: 116 entries, CALCULATION_METHOD_ID to target
dtypes: float64(97), int64(5), object(14)
memory usage: 18.6+ MB


In [964]:
print(df.isna().sum()[df.isna().sum() > 0])

Series([], dtype: int64)


### 1. Первинний огляд
Імпортуємо бібліотеки, завантажуємо датасет та проводимо первинний аудит:
* Перевіряємо розмірність та типи даних (`info()`).
* Шукаємо пропущені значення (`isna()`).
* Видаляємо технічні ідентифікатори (`user_hash`), які не несуть аналітичної цінності.

In [965]:
df = df.drop('user_hash', axis=1)
df = df.drop('CALCULATION_METHOD_ID', axis=1)

print(df['device_model'].value_counts())

df = df.drop(columns='device_model', axis=1)

device_model
0                    1779
Redmi 4X              419
iPhone 6S (A1688)     316
SM-J510H DS           291
iPhone 7 (A1778)      267
                     ... 
SM-N910F                1
SM-J810F DS             1
Xperia SP M35h          1
MI 3                    1
SGH-B320                1
Name: count, Length: 2114, dtype: int64


In [966]:
print(df['Oblast_post_HOME'].value_counts())

Oblast_post_HOME
Київ                    3041
Харківська              2132
Дніпропетровська        1640
0                       1295
Краматорська філія      1094
Запорізька              1073
Київська                1065
Одеська                 1062
Луганська                922
Полтавська               873
Херсонська               838
Донецька                 747
Сумська                  639
Львівська                482
Миколаївська             456
Волинська                452
Кіровоградська           446
Закарпатська             430
Івано-Франківська        405
Чернівецька              385
Чернігівська             331
Сєвєродонецька філія     326
Черкаська                198
Рівненська               182
Вінницька                118
Житомирська              110
Тернопільська            104
Криворізька філія         85
Хмельницька               69
Name: count, dtype: int64


In [967]:
print(df['software_os_vendor'].value_counts())

software_os_vendor
Google         11788
0               3458
Apple           2597
Nokia           1772
Samsung          770
Microsoft        243
Symbian          174
Alibaba           83
Enea              48
Mentor Grap       27
Blackberry        14
Spreadtrum        12
Mediatek          10
Palm               3
MeeGo              1
Name: count, dtype: int64


In [968]:
print(df['device_brand'].value_counts())

device_brand
Samsung                 5208
Apple                   2597
Nokia                   2517
Xiaomi                  2517
0                       1779
                        ... 
Daxian                     1
Fibocom                    1
Bilisim Teknolojiler       1
Anycool                    1
Hongjia                    1
Name: count, Length: 182, dtype: int64


In [969]:
target_values = [0, '0']

mask = (
    df['Oblast_post_HOME'].isin(target_values) &
    df['Raion_post_HOME'].isin(target_values) &
    df['City_post_HOME'].isin(target_values)
)

print(mask.sum())

1295


### 2. Очищення категоріальних ознак
Аналіз показав наявність "сміттєвих" значень (наприклад, '0') у полях регіону проживання та виробника ОС.
* Видаляємо записи з некоректною географією (`Oblast_post_HOME`).
* Видаляємо записи з невизначеним вендором ОС (`software_os_vendor`), оскільки це критично для визначення типу телефону.

In [970]:
df = df[df['Oblast_post_HOME'] != '0']
df = df[df['Oblast_post_HOME'] != 0]

In [971]:
df = df[df['software_os_vendor'] != '0']
df = df[df['software_os_vendor'] != 0]

### 3. Відновлення пропусків у `sim_count`
Кількість SIM-карт часто не визначена (0). Застосовуємо такий підхід:
1. **Apple-коригування:** Якщо бренд Apple/iPhone - ставимо 1 SIM (історично коректно для старих моделей).
2. **Mode Imputation:** Для всіх інших невизначених випадків ставимо 2 SIM (стандарт для ринку України/Android).
3. Виправляємо аномалії (3-4 сімки зводимо до 2).

In [972]:
print(df['sim_count'].value_counts())
print('\n')
mask_zero_sim = df['sim_count'] == 0

mask_apple = mask_zero_sim & (df['device_brand'].astype(str).str.lower().str.contains('apple|iphone|ipad'))
df.loc[mask_apple, 'sim_count'] = 1.0

df.loc[df['sim_count'] == 0, 'sim_count'] = 2.0
df.loc[df['sim_count'] == 3, 'sim_count'] = 2.0
df.loc[df['sim_count'] == 4, 'sim_count'] = 2.0

print('Fixed:')
print(df['sim_count'].value_counts())

sim_count
2.0    6410
0.0    5246
1.0    5122
3.0       9
4.0       7
Name: count, dtype: int64


Fixed:
sim_count
2.0    11032
1.0     5762
Name: count, dtype: int64


### 4. Категорії ОС
Створюємо узагальнену змінну `OS_Category`, яка є сильним предиктором віку та доходу:
* **iOS:** Маркер платоспроможності та молодої/середньої аудиторії.
* **Android:** Масовий сегмент.
* **Feature_Phone:** (Symbian, Bada, Nokia, Microsoft) - сильний маркер вікової групи "Seniors".

In [973]:
def map_os_category(vendor):
    v = str(vendor).lower()

    if 'google' in v or 'android' in v:
        return 'Android'
    elif 'apple' in v or 'ios' in v:
        return 'iOS'
    else:
        return 'Feature_Phone'

df['OS_Category'] = df['software_os_vendor'].apply(map_os_category)

df = pd.get_dummies(df, columns=['OS_Category'], prefix='OS', dtype=int)

cols_to_drop = ['software_os_name', 'software_os_vendor', 'software_os_version']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(df['device_type_rus'].value_counts())
df = df.drop(columns='device_type_rus', axis=1)

device_type_rus
smartphone    14921
phone          1873
Name: count, dtype: int64


### 5. Створення водія (`Is_Driver`)
Аналізуємо активність СМС від мереж АЗС (OKKO, SHELL, WOG тощо).
* **Гіпотеза:** Наявність транзакцій на заправках вказує на наявність автомобіля.
* Створюємо `Fuel_Activity_Score` та бінарний прапор `Is_Driver`. Це важлива ознака для виділення дорослої аудиторії (Adults).

In [974]:
print(df['SHELL'].value_counts())
print('\n')
print(df['gas_stations_sms'].value_counts())
fuel_columns = ['OKKO', 'SHELL', 'SUNOIL', 'KLO', 'BRSM', 'AMIC', 'TNK', 'UPG']

df['Brands_Sum'] = df[fuel_columns].fillna(0).sum(axis=1)

df['Fuel_Activity_Score'] = df[['gas_stations_sms', 'Brands_Sum']].max(axis=1)

df['Is_Driver'] = (df['Fuel_Activity_Score'] > 0).astype(int)

cols_to_drop = fuel_columns + ['gas_stations_sms', 'Brands_Sum']
df = df.drop(columns=cols_to_drop)

df['Fuel_Activity_Score'].value_counts()

SHELL
0.0     16551
2.0       127
4.0        42
6.0        12
5.0        11
1.0        11
8.0        10
10.0        5
7.0         4
3.0         4
14.0        3
23.0        2
26.0        1
17.0        1
50.0        1
11.0        1
41.0        1
33.0        1
19.0        1
28.0        1
24.0        1
13.0        1
27.0        1
20.0        1
Name: count, dtype: int64


gas_stations_sms
0.0      13609
1.0        734
2.0        480
3.0        337
4.0        257
5.0        245
6.0        167
9.0        141
7.0        139
8.0        121
10.0       103
11.0        96
12.0        62
13.0        51
16.0        38
15.0        37
14.0        33
17.0        19
18.0        14
20.0        13
19.0        10
22.0        10
25.0         7
21.0         7
24.0         6
26.0         5
28.0         5
34.0         4
32.0         4
23.0         3
33.0         3
35.0         2
58.0         2
27.0         2
29.0         2
56.0         2
42.0         2
44.0         2
45.0         2
30.0         1
92.0         

Fuel_Activity_Score
0.0      13609
1.0        734
2.0        480
3.0        337
4.0        257
5.0        245
6.0        167
9.0        141
7.0        139
8.0        121
10.0       103
11.0        96
12.0        62
13.0        51
16.0        38
15.0        37
14.0        33
17.0        19
18.0        14
20.0        13
19.0        10
22.0        10
25.0         7
21.0         7
24.0         6
26.0         5
28.0         5
34.0         4
32.0         4
23.0         3
33.0         3
35.0         2
58.0         2
27.0         2
29.0         2
56.0         2
42.0         2
44.0         2
45.0         2
30.0         1
92.0         1
37.0         1
71.0         1
49.0         1
41.0         1
36.0         1
131.0        1
66.0         1
53.0         1
51.0         1
39.0         1
31.0         1
48.0         1
47.0         1
50.0         1
65.0         1
80.0         1
Name: count, dtype: int64

### 6. Обробка інтернет-активності та подiбного
Аналізуємо трафік додатків.
1. **Видалення мультиколінеарності:** Виявлено кореляцію > 0.9 між `volume` (обсяг) та `count` (кількість). Видаляємо `_count` стовпці, залишаючи `_volume`.
2. **Створення соціальних профілів:** На основі специфічних додатків формуємо прапори:
   * `Is_Gamer` (Steam, Twitch)
   * `Dating_App_User` (Tinder, Badoo)
   * `White_Collar_Worker` (LinkedIn, Office365)
   * `Taxi_User` (Uber)

In [975]:
target_cols = [col for col in df.columns if 'volume' in col.lower() or 'count' in col.lower()]

print(f"Total columns count: {len(target_cols)}")

for col in target_cols:
    print(f"\n{'='*20} {col} {'='*20}")

    zeros_count = df[df[col].isin([0, '0', 0.0])].shape[0]
    zeros_pct = (zeros_count / len(df)) * 100

    print(f"Zero percentage: {zeros_pct:.2f}%")
    print("-" * 10)

    print(df[col].value_counts(dropna=False).head(15))

Total columns count: 58

==================== calls_count_in_weekdays ====================
Zero percentage: 2.70%
----------
calls_count_in_weekdays
0.00    453
0.04    165
1.91    123
0.09    122
0.22    122
0.17    120
0.13    118
0.26    112
0.91    110
0.48    108
0.35    107
0.96    106
0.83    106
0.57    105
1.83    104
Name: count, dtype: int64

==================== calls_count_out_weekdays ====================
Zero percentage: 4.06%
----------
calls_count_out_weekdays
0.00    681
0.04    236
0.09    188
0.13    140
0.26    131
0.17    126
0.22    119
0.39    119
0.35    110
0.43    108
1.87    105
0.48    102
2.17    101
0.70    101
0.65    100
Name: count, dtype: int64

==================== calls_count_in_weekends ====================
Zero percentage: 6.70%
----------
calls_count_in_weekends
0.00    1126
0.13     510
0.38     461
0.75     454
0.50     448
0.25     445
0.88     440
1.13     438
1.00     427
0.63     413
1.75     392
1.50     391
1.63     390
1.25     387
1.38 

In [976]:
base_names = set([c.replace('_volume', '').replace('_count', '')
                  for c in df.columns if '_volume' in c or '_count' in c])

cols_to_drop = []

print("Volume vs Count Correlation Coefficients:")
for base in base_names:
    vol_col = f"{base}_volume"
    cnt_col = f"{base}_count"

    if vol_col in df.columns and cnt_col in df.columns:
        corr = df[[vol_col, cnt_col]].corr().iloc[0, 1]
        print(f"{base}: correlation {corr:.4f}")

        if corr > 0.9:
            cols_to_drop.append(cnt_col)

print(f"\n{len(cols_to_drop)} duplicated Count columns will be dropped")
df = df.drop(columns=cols_to_drop)

gamers_cols = ['steam_volume', 'twitch_volume']
available_gamers = [c for c in gamers_cols if c in df.columns]
if available_gamers:
    df['Gamer_Score'] = df[available_gamers].sum(axis=1)
    df['Is_Gamer'] = (df['Gamer_Score'] > 0).astype(int)

dating_cols = ['tinder_volume', 'badoo_volume']
available_dating = [c for c in dating_cols if c in df.columns]
if available_dating:
    df['Dating_App_User'] = (df[available_dating].sum(axis=1) > 0).astype(int)

work_cols = ['linkedin_volume', 'msoffice365_volume', 'dropbox_volume']
available_work = [c for c in work_cols if c in df.columns]
if available_work:
    df['White_Collar_Worker'] = (df[available_work].sum(axis=1) > 0).astype(int)

taxi_cols = ['uber_volume']
available_taxi = [c for c in taxi_cols if c in df.columns]
if available_taxi:
    df['Taxi_User'] = (df[available_taxi].sum(axis=1) > 0).astype(int)

geek_cols = ['jabber_volume', 'tumblr_volume', 'flickr_volume', 'snapchat_volume']
available_geek = [c for c in geek_cols if c in df.columns]
if available_geek:
    df['Niche_Social_User'] = (df[available_geek].sum(axis=1) > 0).astype(int)

expensive_cols = ['netflix_volume', 'itunes_volume']
available_rich = [c for c in geek_cols if c in df.columns]
if available_rich:
    df['Paid_Content_User'] = (df[available_rich].sum(axis=1) > 0).astype(int)

rare_cols_to_remove = available_gamers + available_dating + available_work + available_taxi + available_geek + available_rich
df = df.drop(columns=rare_cols_to_remove)

print("New columns:", ['Is_Gamer', 'Dating_App_User', 'White_Collar_Worker', 'Taxi_User', 'Niche_Social_User', 'Paid_Content_User'])

Volume vs Count Correlation Coefficients:
uber: correlation 0.9265
whatsapp: correlation 0.3050
itunes: correlation 0.7472
steam: correlation 0.6674
dropbox: correlation 0.4276
badoo: correlation 0.8842
viber: correlation 0.8194
netflix: correlation 0.8609
msoffice365: correlation 0.4962
telegram: correlation 0.3868
skype: correlation 0.1371
flickr: correlation 0.4714
twitch: correlation 0.8282
twitter: correlation 0.8251
jabber: correlation 0.6791
google: correlation 0.4882
fb: correlation 0.6545
gmail: correlation 0.4403
applemaps: correlation 0.7604
tumblr: correlation 0.6965
youtube: correlation 0.7485
snapchat: correlation 0.8356
linkedin: correlation 0.6903
tinder: correlation 0.9377

2 duplicated Count columns will be dropped
New columns: ['Is_Gamer', 'Dating_App_User', 'White_Collar_Worker', 'Taxi_User', 'Niche_Social_User', 'Paid_Content_User']


In [977]:
aggregated_apps = [
    'steam', 'twitch',
    'tinder', 'badoo',
    'linkedin', 'msoffice365','dropbox',
    'uber',
    'jabber', 'tumblr', 'flickr', 'snapchat',
    'netflix', 'itunes'
]

cols_to_cleanup = [c for c in df.columns if any(c.startswith(app) for p in aggregated_apps for app in [p])]

df = df.drop(columns=cols_to_cleanup, errors='ignore')

### 7. Створення метрики `Content Heaviness`
Для основних додатків розраховуємо відношення обсягу трафіку до кількості сесій (`Volume / Count`).
* **Високе значення:** Перегляд відео/фото (YouTube, Instagram).
* **Низьке значення:** Текстові повідомлення (Viber, Telegram).
Це дозволяє розрізняти патерни поведінки молоді (відео) та старшого покоління (текст/дзвінки).

In [978]:
apps = ['fb', 'viber', 'youtube', 'google', 'gmail',
        'skype', 'twitter', 'whatsapp', 'applemaps', 'telegram']

for app in apps:
    vol = f"{app}_volume"
    cnt = f"{app}_count"

    df[f'{app}_Content_Heaviness'] = df[vol] / (df[cnt] + 0.001)

    df = df.drop(columns=[cnt], errors='ignore')

### 8. Фінанси та Банки
Аналіз банківських СМС дозволяє виділити чіткі сегменти:
* **Has_Oschad:** Потужний маркер пенсійного віку.
* **Has_Privat:** Масовий сегмент.
* **Has_Raiffeisen/Has_Alfa:** Бiзнес клас.
* **Bank_Diversity:** Кількість банків, якими користується абонент (активна молодь та дорослі часто мають >1 банку).
Створюємо узагальнений `Financial_Activity_Score`.

In [979]:
bank_cols = ['PRIVAT', 'OSCHADBANK', 'ALFABANK', 'UKRSOTBANK', 'OTP',
             'UKRGASBANK', 'RAIFFEISEN', 'PIVDENNYI', 'IDEABANK',
             'SBERBANK', 'MONOBANK', 'PRAVEXBANK', 'UKRSIB']

results = []

for col in bank_cols:
    total_sms = df[col].sum()

    users_count = (df[col] > 0).sum()

    results.append({
        'Bank': col,
        'Total_SMS_Volume': total_sms,
        'Unique_Users': users_count
    })

bank_stats = pd.DataFrame(results).sort_values(by='Unique_Users', ascending=False)

print(bank_stats)

          Bank  Total_SMS_Volume  Unique_Users
0       PRIVAT          161650.0          9372
1   OSCHADBANK           80986.0          2815
2     ALFABANK           42457.0          2149
6   RAIFFEISEN           81785.0          1410
10    MONOBANK            2518.0           666
4          OTP           22619.0           626
12      UKRSIB           11317.0           490
5   UKRGASBANK           14466.0           407
3   UKRSOTBANK            9254.0           308
8     IDEABANK            3546.0           301
7    PIVDENNYI            3743.0           176
11  PRAVEXBANK             171.0            13
9     SBERBANK               0.0             0


In [980]:
df = df.drop(columns=['SBERBANK'])

bank_cols = ['PRIVAT', 'OSCHADBANK', 'ALFABANK', 'UKRSOTBANK', 'OTP',
             'UKRGASBANK', 'RAIFFEISEN', 'PIVDENNYI', 'IDEABANK',
             'MONOBANK', 'PRAVEXBANK', 'UKRSIB']

df['Banks_Calculated_Sum'] = df[bank_cols].sum(axis=1)
df['Financial_Activity_Score'] = df[['banks_sms_count', 'Banks_Calculated_Sum']].max(axis=1)
df['Bank_Diversity'] = (df[bank_cols] > 0).sum(axis=1)

df['Has_Privat'] = (df['PRIVAT'] > 0).astype(int)
df['Has_Oschad'] = (df['OSCHADBANK'] > 0).astype(int)
df['Has_Alfa'] = (df['ALFABANK'] > 0).astype(int)
df['Has_Raiffeisen'] = (df['RAIFFEISEN'] > 0).astype(int)

cols_to_drop = bank_cols + ['banks_sms_count', 'Banks_Calculated_Sum']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("New columns created:")
print(df[['Financial_Activity_Score', 'Bank_Diversity', 'Has_Privat', 'Has_Oschad', 'Has_Alfa', 'Has_Raiffeisen']].head())
print("\nMulty-bank clients count:")
print(df['Bank_Diversity'].value_counts().sort_index())

New columns created:
   Financial_Activity_Score  Bank_Diversity  Has_Privat  Has_Oschad  Has_Alfa  \
0                       0.0               0           0           0         0   
1                       3.0               2           1           0         1   
3                       3.0               1           1           0         0   
4                       0.0               0           0           0         0   
5                     189.0               1           1           0         0   

   Has_Raiffeisen  
0               0  
1               0  
3               0  
4               0  
5               0  

Multy-bank clients count:
Bank_Diversity
0    4658
1    7254
2    3561
3    1002
4     254
5      56
6       8
7       1
Name: count, dtype: int64


### 9. Commute Distance
Використовуємо координати Дім/Робота для розрахунку відстані поїздки (Haversine formula).
* **Commute_Distance_KM:** Дозволяє виділити працююче населення (маятникова міграція) від студентів (живуть в кампусах) та пенсіонерів (локальна мобільність).
* **Урбанізація:** Додаємо прапори великих міст (`Is_Big_City`) та метрики щільності населення.

In [981]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df['Commute_Distance_KM'] = haversine_distance(
    df['LAT_HOME'].fillna(0), df['LON_HOME'].fillna(0),
    df['LAT_WORK'].fillna(0), df['LON_WORK'].fillna(0)
)

cols_to_drop_geo = [
    'LAT_HOME', 'LON_HOME', 'lat_quad_home', 'lon_quad_home',
    'LAT_WORK', 'LON_WORK', 'lat_quad_work', 'lon_quad_work'
]
df = df.drop(columns=cols_to_drop_geo, errors='ignore')

In [982]:
df['City_post_HOME'] = df['City_post_HOME'].astype(str).str.lower().str.strip()

big_cities = [
    'київ', 'kyiv', 'kiev',
    'харків', 'kharkiv',
    'одеса', 'odesa', 'odessa',
    'дніпро', 'dnipro', 'dnepropetrovsk',
    'львів', 'lviv',
    'запоріжжя', 'zaporizhzhia'
]

df['Is_Big_City'] = df['City_post_HOME'].isin(big_cities).astype(int)

city_counts = df['City_post_HOME'].value_counts()
df['City_Population_Proxy'] = df['City_post_HOME'].map(city_counts)

raion_counts = df['Raion_post_HOME'].astype(str).value_counts()
df['Raion_Density'] = df['Raion_post_HOME'].astype(str).map(raion_counts)

df = df.drop(columns=['City_post_HOME', 'Raion_post_HOME'])

print(df[['Is_Big_City', 'City_Population_Proxy', 'Raion_Density']].head(10))

    Is_Big_City  City_Population_Proxy  Raion_Density
0             0                      8             41
1             1                    670            676
3             0                      3             95
4             0                     32             35
5             1                   1281           1281
6             0                     23             39
7             0                    307            310
8             0                    253            253
10            0                    365            394
11            0                      6              6


### 10. Поведінкові патерни (Voice vs Data)
Створюємо похідні метрики для опису стилю спілкування:
* `Weekend_Calls_Share`: Частка активності у вихідні.
* `Avg_Call_Duration`: Середня тривалість розмови (літні люди схильні говорити довше).
* `Digital_Introvert_Score`: Співвідношення Дата-трафіку до Голосових дзвінків. Ключовий індикатор для розрізнення поколінь.

In [983]:
total_calls = df['calls_count_in_weekdays'] + df['calls_count_out_weekdays'] + \
              df['calls_count_in_weekends'] + df['calls_count_out_weekends']

df['Weekend_Calls_Share'] = (df['calls_count_in_weekends'] + df['calls_count_out_weekends']) / (total_calls + 0.001)

total_data = df['DATA_VOLUME_WEEKDAYS'] + df['DATA_VOLUME_WEEKENDS']
df['Weekend_Data_Share'] = df['DATA_VOLUME_WEEKENDS'] / (total_data + 0.001)

df['Avg_Call_Duration_Weekdays'] = (df['calls_duration_in_weekdays'] + df['calls_duration_out_weekdays']) / \
                                   (df['calls_count_in_weekdays'] + df['calls_count_out_weekdays'] + 0.001)

df['Avg_Call_Duration_Weekends'] = (df['calls_duration_in_weekends'] + df['calls_duration_out_weekends']) / \
                                   (df['calls_count_in_weekends'] + df['calls_count_out_weekends'] + 0.001)

df['Digital_Introvert_Score'] = total_data / (total_calls + 1)

cols_to_drop = ['calls_duration_in_weekdays', 'calls_duration_out_weekdays',
                'calls_duration_in_weekends', 'calls_duration_out_weekends']

df = df.drop(columns=cols_to_drop)

print("Features of clients behaviour have been created. Duration columns have been dropped.")
print(df[['Weekend_Calls_Share', 'Avg_Call_Duration_Weekdays', 'Digital_Introvert_Score']].head())

Features of clients behaviour have been created. Duration columns have been dropped.
   Weekend_Calls_Share  Avg_Call_Duration_Weekdays  Digital_Introvert_Score
0             0.387882                    2.546014                 7.037704
1             0.418235                    1.360173                30.279646
3             0.421139                    4.048719                 0.000000
4             0.665484                    2.303271                 0.000000
5             0.379575                    5.380397                 4.337599


### 11. Обробка брендів та Скорингу
* **Бренди:** Групуємо менш популярні бренди в категорію 'Other' та застосовуємо One-Hot Encoding.
* **Кредитний скоринг (`SCORING`):** Перетворюємо порядкову текстову змінну в числову шкалу (1-4). Пропущені значення заповнюємо модою.

In [984]:
df['device_brand'].value_counts()

device_brand
Samsung           4459
Apple             2481
Xiaomi            2453
Nokia             1775
Lenovo            1042
                  ... 
Symphony             1
MyPhone              1
Minte Telecom        1
Taidian              1
Jurong Hi-Tech       1
Name: count, Length: 133, dtype: int64

In [985]:
import pandas as pd

top_brands = ['Samsung', 'Apple', 'Xiaomi', 'Lenovo', 'Huawei']

df['device_brand_group'] = df['device_brand'].apply(lambda x: x if x in top_brands else 'Other')

df = pd.get_dummies(df, columns=['device_brand_group'], prefix='Brand', dtype=int)

df = df.drop(columns=['device_brand'])

if 'Brand_Apple' in df.columns:
    df = df.drop(columns=['Brand_Apple'])

In [986]:
df['SCORING'].value_counts()

SCORING
HIGH_MEDIUM    5150
MEDIUM         4161
LOW            4044
HIGH           2437
VERY LOW       1001
0                 1
Name: count, dtype: int64

In [987]:
scoring_map = {
    'VERY LOW': 1,
    'LOW': 1,
    'MEDIUM': 2,
    'HIGH_MEDIUM': 3,
    'HIGH': 4,
    '0': np.nan
}

df['SCORING_Numeric'] = df['SCORING'].map(scoring_map)

mode_val = df['SCORING_Numeric'].mode()[0]
df['SCORING_Numeric'] = df['SCORING_Numeric'].fillna(mode_val)

df = df.drop(columns=['SCORING'])

print("Brands have been encoded")
print("Scoring has been converted to numbers:")
print(df['SCORING_Numeric'].value_counts().sort_index())

Brands have been encoded
Scoring has been converted to numbers:
SCORING_Numeric
1.0    5045
2.0    4161
3.0    5151
4.0    2437
Name: count, dtype: int64


In [988]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16794 entries, 0 to 20999
Data columns (total 75 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   calls_count_in_weekdays      16794 non-null  float64
 1   calls_count_out_weekdays     16794 non-null  float64
 2   calls_count_in_weekends      16794 non-null  float64
 3   calls_count_out_weekends     16794 non-null  float64
 4   DATA_VOLUME_WEEKDAYS         16794 non-null  float64
 5   DATA_VOLUME_WEEKENDS         16794 non-null  float64
 6   Oblast_post_HOME             16794 non-null  object 
 7   Oblast_post_WORK             16794 non-null  object 
 8   Raion_post_WORK              16794 non-null  object 
 9   City_post_WORK               16794 non-null  object 
 10  sim_count                    16794 non-null  float64
 11  AVG_ARPU                     16794 non-null  float64
 12  ROUM                         16794 non-null  float64
 13  phone_value          

### 12. Робота та Регіональна щільність
* Обробляємо локацію роботи (`City_post_WORK`): виділяємо великі міста та щільність районів.
* Замінюємо текстові назви областей (`Oblast_post`) на частотні значення (Frequency Encoding).

In [989]:
big_cities = [
    'київ', 'kyiv', 'kiev',
    'харків', 'kharkiv',
    'одеса', 'odesa', 'odessa',
    'дніпро', 'dnipro', 'dnepropetrovsk',
    'львів', 'lviv',
    'запоріжжя', 'zaporizhzhia'
]

df['City_post_WORK'] = df['City_post_WORK'].astype(str).str.lower().str.strip()

df['Is_Big_City_WORK'] = df['City_post_WORK'].isin(big_cities).astype(int)

work_city_counts = df['City_post_WORK'].value_counts()
df['City_Population_Proxy_WORK'] = df['City_post_WORK'].map(work_city_counts)

work_raion_counts = df['Raion_post_WORK'].astype(str).value_counts()
df['Raion_Density_WORK'] = df['Raion_post_WORK'].astype(str).map(work_raion_counts)

In [990]:
oblast_home_counts = df['Oblast_post_HOME'].value_counts()
df['Oblast_Density_HOME'] = df['Oblast_post_HOME'].map(oblast_home_counts)

oblast_work_counts = df['Oblast_post_WORK'].value_counts()
df['Oblast_Density_WORK'] = df['Oblast_post_WORK'].map(oblast_work_counts)

text_geo_cols = ['Oblast_post_HOME', 'Oblast_post_WORK', 'Raion_post_WORK', 'City_post_WORK']
df = df.drop(columns=text_geo_cols)

### 13. Створення фінальних мета-ознак та Експорт
Додаємо синтетичні ознаки, які показали найвищу кореляцію з віком на етапі аналізу:
1. **`Tech_Refresh_Rate`:** Відношення часу володіння телефоном до загального стажу в мережі. (Консерватори vs Новатори).
2. **`Affluence_Index`:** Комбінація ARPU та вартості телефону. Дозволяє виділити найбільш платоспроможне ядро (35-45 років).

Зберігаємо чистий датасет (`cleaned_data.csv`).

In [991]:
df['Tech_Refresh_Rate'] = df['how_long_same_model'] / (df['lifetime'] + 1)

In [992]:
df['Affluence_Index'] = df['AVG_ARPU'] * (5 - df['phone_value'])

In [993]:
print("Text columns count:", df.select_dtypes(include=['object']).shape[1])
print(f"Final shape: {df.shape}")

Text columns count: 0
Final shape: (16794, 78)


In [994]:
df.to_csv('.\\cleaned_data.csv', index=False)